# 07b — Consultas espaciales MT: relaciones entre capas

Segundo bloque corto: vecino más cercano, relación cable-poste y ventana de trabajo.

> Esta notebook es una parte del bloque 07. Está separada para que sea más liviana de editar, ejecutar y revisar en clase.

## 1. Objetivo y prerequisitos

- Tener levantado PostGIS con `docker compose up -d postgis`.
- Haber ejecutado la notebook 06 para publicar `gis.mt_cables`, `gis.mt_postes` y `gis.mt_seccionadores`.
- Ejecutar las celdas en orden. Cada consulta deja el SQL visible antes del resultado.

## 2. Preparación de conexión

In [ ]:
# pathlib permite ubicar la raíz del proyecto aunque Jupyter se abra desde otra carpeta.
from pathlib import Path

# os permite leer variables de entorno para parametrizar la conexión local a PostGIS.
import os

# pandas se usa solo para mostrar resultados tabulares de forma cómoda en Jupyter.
import pandas as pd

# psycopg es el driver moderno de PostgreSQL usado por el proyecto.
try:
    import psycopg
    PSYCOPG_DISPONIBLE = True
except ModuleNotFoundError:
    psycopg = None
    PSYCOPG_DISPONIBLE = False
    print('No está instalado el paquete psycopg en este kernel.')
    print('Ejecutar desde la raíz del proyecto: python3 -m pip install -r requirements.txt')
    print('Después de instalar, reiniciar el kernel de Jupyter y volver a ejecutar la notebook.')

# Buscamos la raíz del proyecto subiendo hasta encontrar archivos propios del repo.
RAIZ = Path.cwd()
while RAIZ.parent != RAIZ:
    if (RAIZ / 'docker-compose.yml').exists() and (RAIZ / 'scripts' / 'gis_sqlserver.sh').exists():
        break
    RAIZ = RAIZ.parent

# Cargamos .env local sin pisar variables ya definidas por el entorno.
def cargar_env_local() -> None:
    env_path = RAIZ / '.env'
    if not env_path.exists():
        return
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip())

cargar_env_local()

DB_CONFIG = {
    'host': os.getenv('POSTGRES_HOST', 'localhost'),
    'port': int(os.getenv('POSTGRES_PORT', '5432')),
    'dbname': os.getenv('POSTGRES_DB', 'ceml_gis'),
    'user': os.getenv('POSTGRES_USER', 'ceml'),
    'password': os.getenv('POSTGRES_PASSWORD', 'ceml_admin_2026'),
}

SRID_PUBLICACION = 32721

# Esta función corta con una instrucción clara si falta el driver de PostgreSQL.
def exigir_psycopg() -> None:
    if not PSYCOPG_DISPONIBLE:
        raise RuntimeError(
            'Falta instalar psycopg en este kernel. '
            'Ejecutar: python -m pip install -r requirements.txt, '
            'reiniciar el kernel y volver a correr la notebook.'
        )

# Ejecuta una consulta SELECT y devuelve un DataFrame.
# El autocommit evita dejar transacciones abiertas durante ejercicios de solo lectura.
def ejecutar_sql(sql: str) -> pd.DataFrame:
    exigir_psycopg()
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            columnas = [col.name for col in cur.description]
            filas = cur.fetchall()
    return pd.DataFrame(filas, columns=columnas)

print('Raíz del proyecto:', RAIZ)
print('Base local:', DB_CONFIG['dbname'])
print('Host local:', DB_CONFIG['host'])
print('Puerto local:', DB_CONFIG['port'])
print('Usuario local:', DB_CONFIG['user'])
print('SRID de publicación:', SRID_PUBLICACION)

## 3. Verificación de capas GIS

In [ ]:
# Verificamos que estén publicadas las capas finales que usan estas consultas.
SQL_VERIFICAR_CAPAS = """
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_schema = 'gis'
  AND table_name IN ('mt_cables', 'mt_postes', 'mt_seccionadores')
ORDER BY table_name;
"""

ejecutar_sql(SQL_VERIFICAR_CAPAS)

## 4. Consulta 4: Cable más cercano a cada seccionador

**Consulta espacial:** Vecino más cercano  
**Función:** Asociar cada seccionador con el cable más próximo.  
**Consigna:** Para cada seccionador, obtener el cable más cercano y la distancia entre ambos.

**Idea clave:** Para cada seccionador busca el cable más cercano usando el operador espacial <->, útil para consultas de vecino más próximo.

In [ ]:
SQL_CONSULTA_04 = """
SELECT
    s.id AS id_seccionador,
    s.idcad AS idcad_seccionador,
    c.id AS id_cable,
    c.idcad AS idcad_cable,
    round(ST_Distance(s.geom, c.geom)::numeric, 2) AS distancia_m
FROM gis.mt_seccionadores s
CROSS JOIN LATERAL (
    SELECT
        c.id,
        c.idcad,
        c.geom
    FROM gis.mt_cables c
    ORDER BY s.geom <-> c.geom
    LIMIT 1
) c
ORDER BY distancia_m;
"""

print(SQL_CONSULTA_04)

In [ ]:
ejecutar_sql(SQL_CONSULTA_04)

## 5. Consulta 5: Cantidad de postes cercanos a cada cable

**Consulta espacial:** Relación por cercanía ST_DWithin  
**Función:** Contar postes próximos a cada tramo de cable.  
**Consigna:** Listar los cables y contar cuántos postes se encuentran a menos de 10 metros de cada cable.

**Idea clave:** Relaciona cables y postes cuando la distancia entre ambos es menor o igual a 10 metros. Sirve para detectar qué tramos tienen más postes asociados espacialmente.

In [ ]:
SQL_CONSULTA_05 = """
SELECT
    c.id,
    c.idcad,
    c.coop,
    count(p.id) AS postes_cercanos
FROM gis.mt_cables c
JOIN gis.mt_postes p
    ON ST_DWithin(p.geom, c.geom, 10)
GROUP BY c.id, c.idcad, c.coop
ORDER BY postes_cercanos DESC;
"""

print(SQL_CONSULTA_05)

In [ ]:
ejecutar_sql(SQL_CONSULTA_05)

## 6. Consulta 6: Postes dentro de un rectángulo de trabajo

**Consulta espacial:** Contención espacial ST_Within  
**Función:** Filtrar elementos dentro de un área rectangular.  
**Consigna:** Obtener los postes contenidos dentro de una ventana espacial definida en coordenadas UTM EPSG:32721.

**Idea clave:** ST_MakeEnvelope crea un rectángulo espacial. ST_Within verifica qué postes están completamente contenidos dentro de esa geometría.

In [ ]:
SQL_CONSULTA_06 = """
SELECT
    p.id,
    p.idcad,
    p.coop,
    p.capa_origen
FROM gis.mt_postes p
WHERE ST_Within(
    p.geom,
    ST_MakeEnvelope(720000, 7040000, 725000, 7045000, 32721)
);
"""

print(SQL_CONSULTA_06)

In [ ]:
ejecutar_sql(SQL_CONSULTA_06)

## Cierre

Si este bloque ejecutó correctamente, continuá con la siguiente parte 07. La separación evita notebooks largas y facilita retomar desde el punto exacto donde quedó la clase.